# Day 13 — CuTe DSL TV-Layout Elementwise + Predication

**Goal:** rewrite day 12 using an explicit Thread-Value layout plus a
coordinate tensor + predicate, so the kernel handles `M`/`N` that aren't
multiples of the CTA tile.

This is **the** canonical CuTe DSL pattern. Five steps:

```
1. blkA   = gA[((None, None), bidx)]          tile per CTA
   blkCrd = cC[((None, None), bidx)]          coord tile per CTA
2. tidfrgA   = composition(blkA, tv_layout)   (tid, vid) -> addr
   tidfrgCrd = composition(blkCrd, tv_layout) (tid, vid) -> coord
3. thrA   = tidfrgA[(tidx, None)]             vid -> addr
   thrCrd = tidfrgCrd[(tidx, None)]
4. frgPred[i] = elem_less(thrCrd[i], shape)
5. predicated load -> compute -> predicated store
```


In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
import torch

cutlass.cuda.initialize_cuda_context()


## The kernel

The host code is filled in — implement the five-step kernel body.


In [ ]:
@cute.kernel
def elementwise_add_kernel(gA, gB, gC, cC, shape, tv_layout):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    # TODO: implement steps 1-5. See the cells below for the full solution.
    raise NotImplementedError


@cute.jit
def elementwise_add(mA, mB, mC):
    assert mA.element_type == mB.element_type == mC.element_type
    dtype = mA.element_type
    elts_per_vec: cutlass.Constexpr = 128 // dtype.width
    thr_layout = cute.make_ordered_layout((4, 32), order=(1, 0))
    val_layout = cute.make_ordered_layout((4, elts_per_vec), order=(1, 0))
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)
    gA = cute.zipped_divide(mA, tiler_mn)
    gB = cute.zipped_divide(mB, tiler_mn)
    gC = cute.zipped_divide(mC, tiler_mn)
    idC = cute.make_identity_tensor(mC.shape)
    cC = cute.zipped_divide(idC, tiler=tiler_mn)
    elementwise_add_kernel(gA, gB, gC, cC, mC.shape, tv_layout).launch(
        grid=(cute.size(gC, mode=[1]), 1, 1),
        block=(cute.size(tv_layout, mode=[0]), 1, 1),
    )


# M, N = 1023, 1025   # deliberately non-tile-multiple
# a = torch.randn(M, N, device="cuda", dtype=torch.float16)
# b = torch.randn(M, N, device="cuda", dtype=torch.float16)
# c = torch.zeros_like(a)
# a_ = from_dlpack(a, assumed_align=16).mark_layout_dynamic()
# b_ = from_dlpack(b, assumed_align=16).mark_layout_dynamic()
# c_ = from_dlpack(c, assumed_align=16).mark_layout_dynamic()
# elementwise_add(a_, b_, c_)
# torch.cuda.synchronize()
# torch.testing.assert_close(c, a + b); print("OK")


---

## Reference solution


In [ ]:
@cute.kernel
def elementwise_add_kernel(gA, gB, gC, cC, shape, tv_layout):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # 1. CTA tile
    blk_coord = ((None, None), bidx)
    blkA, blkB, blkC = gA[blk_coord], gB[blk_coord], gC[blk_coord]
    blkCrd = cC[blk_coord]

    # 2. compose with TV layout: (tid, vid) -> addr / coord
    tidfrgA   = cute.composition(blkA,   tv_layout)
    tidfrgB   = cute.composition(blkB,   tv_layout)
    tidfrgC   = cute.composition(blkC,   tv_layout)
    tidfrgCrd = cute.composition(blkCrd, tv_layout)

    # 3. slice down to this thread
    thr_coord = (tidx, None)
    thrA, thrB, thrC = tidfrgA[thr_coord], tidfrgB[thr_coord], tidfrgC[thr_coord]
    thrCrd = tidfrgCrd[thr_coord]

    # 4. build predicate fragment
    frgPred = cute.make_fragment(thrCrd.shape, cutlass.Boolean)
    for i in cutlass.range_constexpr(cute.size(frgPred)):
        frgPred[i] = cute.elem_less(thrCrd[i], shape)

    # 5. predicated load -> compute -> predicated store
    frgA = cute.make_fragment_like(thrA)
    frgB = cute.make_fragment_like(thrB)
    frgC = cute.make_fragment_like(thrC)
    for i in cutlass.range_constexpr(cute.size(frgA)):
        if frgPred[i]:
            frgA[i] = thrA[i]
            frgB[i] = thrB[i]
    frgC.store(frgA.load() + frgB.load())
    for i in cutlass.range_constexpr(cute.size(frgC)):
        if frgPred[i]:
            thrC[i] = frgC[i]


M, N = 1023, 1025
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros_like(a)
a_ = from_dlpack(a, assumed_align=16).mark_layout_dynamic()
b_ = from_dlpack(b, assumed_align=16).mark_layout_dynamic()
c_ = from_dlpack(c, assumed_align=16).mark_layout_dynamic()
elementwise_add(a_, b_, c_)
torch.cuda.synchronize()
torch.testing.assert_close(c, a + b); print("OK on (1023, 1025)")
